# Embedding Retrieval Prototype (V2)

Dense retrieval with `intfloat/multilingual-e5-base`, evaluated on the same `dev_queries.json` and metrics as V1 (`retrieval_baseline.ipynb`).

## 1. Load data, embeddings and model

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer

DATA = Path("../data")
pois = gpd.read_parquet(DATA / "processed" / "pois_v2.parquet")
E = np.load(DATA / "processed" / "poi_embeddings.npy")
dev = json.loads((DATA / "eval" / "dev_queries.json").read_text(encoding="utf-8"))
assert len(pois) == E.shape[0], "pois_v2.parquet and poi_embeddings.npy are out of sync"

IDS, LAT, LON = pois["id"].to_numpy(), pois["lat"].to_numpy(), pois["lon"].to_numpy()
missing = {i for q in dev for i in q["expected_ids"]} - set(IDS.tolist())
print(f"POIs: {len(pois):,}, embeddings: {E.shape}, dev queries: {len(dev)}")
print(f"Expected ids missing from pois_v2: {len(missing)}")

model = SentenceTransformer("intfloat/multilingual-e5-base")

c:\Users\Korisnik\GeoIR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


POIs: 44,149, embeddings: (44149, 768), dev queries: 50
Expected ids missing from pois_v2: 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 279.58it/s]


## 2. Search

Score = cosine similarity (`E @ q`); with coordinates, candidates are limited to `radius_km` and a distance bonus is added.

In [2]:
SPATIAL_WEIGHT = 0.1   # weight of the distance bonus (1 at the centre, 0 at the radius edge)

def encode_queries(texts):
    return model.encode(["query: " + t for t in texts], normalize_embeddings=True, convert_to_numpy=True)

def haversine_km(lat, lon):
    p1, p2 = np.radians(lat), np.radians(LAT)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(LON - lon) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def search(q_vec, lat=None, lon=None, radius_km=None, spatial_weight=SPATIAL_WEIGHT):
    sim = E @ q_vec
    if lat is not None and lon is not None and radius_km is not None:
        d = haversine_km(lat, lon)
        cand = np.flatnonzero(d <= radius_km)
        score = sim[cand] + spatial_weight * (1 - d[cand] / radius_km)
        order = np.argsort(-score, kind="stable")
        return cand[order], score[order]
    order = np.argsort(-sim, kind="stable")
    return order, sim[order]

Q = encode_queries([q["query"] for q in dev])
for q, qv in list(zip(dev, Q))[::12]:
    pos, sc = search(qv, q["lat"], q["lon"], q["radius_km"])
    print(f"{q['query_id']} {q['query']!r}")
    for p, s in zip(pos[:3], sc[:3]):
        print(f"   {s:.3f} {'✓' if IDS[p] in q['expected_ids'] else '✗'} {pois.at[p, 'nlp_text'][:100]}")

dev-ent-01 'flower u Mostaru'
   0.880 ✗ Cvjećara EDO, florist, Mostar, Grad Mostar
   0.880 ✗ Cvjećara JASMINN, florist, Mostar, Grad Mostar
   0.879 ✗ florist, Mostar, Grad Mostar
dev-ent-13 'alpha plus adresa'
   0.861 ✓ Alpha Plus, pharmacy, Dr. Ante Starčevića 54, Mostar, 88000
   0.835 ✗ Sava plus, convenience, H. H. Jerkovića 23, Brčko, 76100
   0.832 ✗ Profit Plus, accountant, Branilaca Šipa 43, Sarajevo, Općina Centar, 71000
dev-spa-10 'najbliži dječije igralište'
   0.870 ✗ atm, Addiko Bank, Tuzla, Grad Tuzla
   0.869 ✗ Žuta kuća, fast food, Soni trg, Tuzla, Grad Tuzla
   0.868 ✗ marketplace, Tuzla, Grad Tuzla
dev-des-07 'gdje mogu naći brzu hranu sa bežičnim internetom i gdje mogu platiti karticom u Sarajevu'
   0.860 ✗ BA, cafe, payment cash, payment credit cards, payment debit cards, outdoor seating, Put života 8, Sa
   0.860 ✗ Chickies, fast food, chicken, Sarajevo, Općina Centar, Grad Sarajevo
   0.860 ✗ Alibaba, fast food, turkish, Sarajevo, Općina Novo Sarajevo, Grad S

## 3. Metrics

Same definitions as V1: Hit@K, Recall@K, Precision@5, MRR, NDCG@K (binary relevance).

In [3]:
def hit_at_k(ranked, rel, k):
    return float(any(r in rel for r in ranked[:k]))

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & rel) / len(rel)

def precision_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & rel) / k

def reciprocal_rank(ranked, rel):
    for i, r in enumerate(ranked, start=1):
        if r in rel:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked, rel, k):
    dcg = sum(1.0 / np.log2(i + 1) for i, r in enumerate(ranked[:k], start=1) if r in rel)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
    return dcg / idcg

METRICS = ["Hit@5", "Hit@10", "Recall@5", "Recall@10", "P@5", "MRR", "NDCG@5", "NDCG@10"]
TYPE_ORDER = ["entity", "spatial_category", "descriptive", "cross_lingual"]

def evaluate(spatial_weight=SPATIAL_WEIGHT):
    rows = []
    for q, qv in zip(dev, Q):
        rel = set(q["expected_ids"])
        pos, _ = search(qv, q["lat"], q["lon"], q["radius_km"], spatial_weight)
        ranked = IDS[pos[:1000]].tolist()
        rows.append({"query_id": q["query_id"], "type": q["type"], "query": q["query"],
                     "Hit@5": hit_at_k(ranked, rel, 5), "Hit@10": hit_at_k(ranked, rel, 10),
                     "Recall@5": recall_at_k(ranked, rel, 5), "Recall@10": recall_at_k(ranked, rel, 10),
                     "P@5": precision_at_k(ranked, rel, 5), "MRR": reciprocal_rank(ranked, rel),
                     "NDCG@5": ndcg_at_k(ranked, rel, 5), "NDCG@10": ndcg_at_k(ranked, rel, 10)})
    return pd.DataFrame(rows)

res = evaluate()

## 4. Results by query type

In [4]:
v2_type = res.groupby("type")[METRICS].mean().reindex(TYPE_ORDER)
v2_type.loc["overall"] = res[METRICS].mean()
display(v2_type.round(3))

,Hit@5,Hit@10,Recall@5,Recall@10,P@5,MRR,NDCG@5,NDCG@10
type,,,,,,,,
entity,0.600,0.667,0.533,0.667,0.120,0.516,0.482,0.534
spatial_category,0.667,0.667,0.181,0.264,0.440,0.626,0.483,0.423
descriptive,0.083,0.167,0.083,0.104,0.017,0.103,0.083,0.094
cross_lingual,0.625,1.000,0.380,0.593,0.225,0.503,0.385,0.482
overall,0.500,0.600,0.295,0.399,0.208,0.448,0.371,0.387


## 6. Spatial weight check

Spatial queries with the distance bonus vs. radius filter only.

In [5]:
no_bonus = evaluate(spatial_weight=0.0)
sp = lambda r: r[r["type"] == "spatial_category"][METRICS].mean()
display(pd.DataFrame({"filter only": sp(no_bonus), f"filter + bonus ({SPATIAL_WEIGHT})": sp(res)}).T.round(3))

,Hit@5,Hit@10,Recall@5,Recall@10,P@5,MRR,NDCG@5,NDCG@10
filter only,0.867,1.000,0.403,0.627,0.72,0.884,0.797,0.818
filter + bonus (0.1),0.667,0.667,0.181,0.264,0.44,0.626,0.483,0.423


## 7. Failures (Hit@5 = 0)

In [6]:
fails = res[res["Hit@5"] == 0][["query_id", "type", "query", "MRR"]]
print(f"{len(fails)} / {len(res)} dev queries without a relevant POI in the top 5")
display(fails.round(3))

25 / 50 dev queries without a relevant POI in the top 5


,query_id,type,query,MRR
0,dev-ent-01,entity,flower u Mostaru,0.016
2,dev-ent-03,entity,djulbasca adresa,0.056
4,dev-ent-05,entity,dzamija kalesija adresa,0.167
5,dev-ent-06,entity,euro adresa,0.038
6,dev-ent-07,entity,kripton adresa,0.004
9,dev-ent-10,entity,donji zelenikovac adresa,0.056
15,dev-spa-01,spatial_category,džamija blizu mene,0.004
16,dev-spa-02,spatial_category,benzinska pumpa blizu mene,0.009
18,dev-spa-04,spatial_category,najbliža benzinska pumpa,0.019
24,dev-spa-10,spatial_category,najbliži dječije igralište,0.012
